# FPGA Metrics
Run all cells. This notebook measures slice timing, simple registration losses, and board power if PYNQ exposes the rails.

In [ ]:
import os
import json
import time
import threading
from pathlib import Path

import cv2
import matplotlib.pyplot as plt
import numpy as np
from pynq import allocate
from pynq_dpu import DpuOverlay
import xir
import vart


In [ ]:
cfg_path = Path('metrics_config.json')
if cfg_path.exists():
    cfg = json.loads(cfg_path.read_text())
else:
    cfg = {
        'bitstream': 'dpu.bit',
        'xmodel': './vxm_2p5d_pt_v3.xmodel',
        'mr_volume': './1BA001_mr.npy',
        'ct_volume': './1BA005_ct.npy',
        'mr_seg': './1BA001_mr_seg.npy',
        'ct_seg': './1BA005_ct_seg.npy',
        'input_hw': [112, 96],
        'window_radius': 3,
        'n_stack': 8,
        'seg_labels': [2, 3, 4, 7, 8, 10, 11, 12, 13, 14, 15, 16, 17, 18, 26, 28],
        'output_dir': './fpga_metrics',
        'power_period_s': 0.1,
        'preferred_power_rail': 'PSINT_FP',
        'warmup_runs': 3,
        'quiver_flip_y': True,
    }

def resolve_file(path_str):
    p = Path(path_str)
    if p.exists():
        return str(p)
    for rel in ('./data/test_data', './test_data'):
        alt = Path(rel) / p.name
        if alt.exists():
            return str(alt)
    return str(p)

DPU_BITSTREAM = cfg['bitstream']
XMODEL_PATH = resolve_file(cfg['xmodel'])
MR_VOLUME_PATH = resolve_file(cfg['mr_volume'])
CT_VOLUME_PATH = resolve_file(cfg['ct_volume'])
MR_SEG_PATH = resolve_file(cfg['mr_seg'])
CT_SEG_PATH = resolve_file(cfg['ct_seg'])
INPUT_HEIGHT, INPUT_WIDTH = [int(x) for x in cfg['input_hw']]
WINDOW_RADIUS = int(cfg['window_radius'])
N_STACK = int(cfg['n_stack'])
SEG_LABELS = [int(x) for x in cfg['seg_labels']]
OUTPUT_DIR = cfg.get('output_dir', './fpga_metrics')
POWER_PERIOD_S = float(cfg.get('power_period_s', 0.1))
PREFERRED_POWER_RAIL = cfg.get('preferred_power_rail')
WARMUP_RUNS = int(cfg.get('warmup_runs', 3))
QUIVER_FLIP_Y = bool(cfg.get('quiver_flip_y', True))

os.makedirs(OUTPUT_DIR, exist_ok=True)

print('XMODEL:', XMODEL_PATH)
print('Moving:', MR_VOLUME_PATH)
print('Fixed:', CT_VOLUME_PATH)
print('Input:', (INPUT_HEIGHT, INPUT_WIDTH))
print('Stacks:', N_STACK)
print('Output dir:', OUTPUT_DIR)


In [ ]:
overlay = DpuOverlay(DPU_BITSTREAM)
graph = xir.Graph.deserialize(XMODEL_PATH)
subgraphs = graph.get_root_subgraph().toposort_child_subgraph()
dpu_subgraph = None
for subgraph in subgraphs:
    if subgraph.has_attr('device') and subgraph.get_attr('device') == 'DPU':
        dpu_subgraph = subgraph
        break
if dpu_subgraph is None:
    raise RuntimeError('No DPU subgraph found')

dpu_runner = vart.Runner.create_runner(dpu_subgraph, 'run')
input_tensors = dpu_runner.get_input_tensors()
output_tensors = dpu_runner.get_output_tensors()
output_shape = tuple(output_tensors[0].dims)

FIXED_SCALE = None
MOVING_SCALE = None
for tensor in input_tensors:
    scale = float(2 ** int(tensor.get_attr('fix_point')))
    name = str(tensor.name).lower()
    if 'fixed' in name:
        FIXED_SCALE = scale
    elif 'moving' in name:
        MOVING_SCALE = scale

if len(input_tensors) == 1:
    FIXED_SCALE = FIXED_SCALE or float(2 ** int(input_tensors[0].get_attr('fix_point')))
    MOVING_SCALE = MOVING_SCALE or FIXED_SCALE
else:
    if FIXED_SCALE is None:
        FIXED_SCALE = float(2 ** int(input_tensors[0].get_attr('fix_point')))
    if MOVING_SCALE is None:
        MOVING_SCALE = float(2 ** int(input_tensors[-1].get_attr('fix_point')))

OUTPUT_SCALE = float(2 ** int(output_tensors[0].get_attr('fix_point')))

for tensor in input_tensors:
    print('input', tensor.name, list(tensor.dims), 'scale', float(2 ** int(tensor.get_attr('fix_point'))))
for tensor in output_tensors:
    print('output', tensor.name, list(tensor.dims), 'scale', float(2 ** int(tensor.get_attr('fix_point'))))


In [ ]:
def extract_slice_stack(volume, axis, z, window_radius, target_size=(112, 96), nearest=False):
    wr = window_radius
    if axis == 0:
        stack = volume[z - wr:z + wr + 1]
    elif axis == 1:
        stack = volume[:, z - wr:z + wr + 1, :].transpose(1, 0, 2)
    else:
        stack = volume[:, :, z - wr:z + wr + 1].transpose(2, 0, 1)
    interp = cv2.INTER_NEAREST if nearest else cv2.INTER_LINEAR
    resized = [cv2.resize(s, (target_size[1], target_size[0]), interpolation=interp) for s in stack]
    return np.ascontiguousarray(np.array(resized, dtype=np.float32))


def pad_stack_to_n(stack, n_stack):
    if stack.shape[0] == n_stack:
        return stack
    if stack.shape[0] > n_stack:
        return stack[:n_stack]
    out = stack.copy()
    while out.shape[0] < n_stack:
        out = np.concatenate([out, out[-1:]], axis=0)
    return out


def resize_flow(flow, out_hw):
    out_h, out_w = out_hw
    if flow.shape[1:] == (out_h, out_w):
        return flow
    in_h, in_w = flow.shape[1:]
    flow_x = cv2.resize(flow[0], (out_w, out_h), interpolation=cv2.INTER_LINEAR)
    flow_y = cv2.resize(flow[1], (out_w, out_h), interpolation=cv2.INTER_LINEAR)
    flow_x *= float(out_w) / float(in_w)
    flow_y *= float(out_h) / float(in_h)
    return np.stack([flow_x, flow_y], axis=0).astype(np.float32)


def warp_image(image, flow, nearest=False):
    h, w = image.shape
    flow = resize_flow(flow, (h, w))
    y, x = np.mgrid[0:h, 0:w].astype(np.float32)
    new_x = x + flow[0]
    new_y = y + flow[1]
    interp = cv2.INTER_NEAREST if nearest else cv2.INTER_LINEAR
    return cv2.remap(image.astype(np.float32), new_x, new_y, interp, borderMode=cv2.BORDER_CONSTANT).astype(np.float32)


def dice_score(pred_seg, gt_seg, labels=SEG_LABELS):
    vals = []
    for label in labels:
        pred = (pred_seg == label).astype(np.float32)
        gt = (gt_seg == label).astype(np.float32)
        gt_sum = float(np.sum(gt))
        if gt_sum == 0.0:
            continue
        inter = float(np.sum(pred * gt))
        union = float(np.sum(pred) + gt_sum)
        vals.append((2.0 * inter + 1e-5) / (union + 1e-5))
    return float(np.mean(vals)) if vals else None


def mutual_info(a, b, bins=32):
    a = np.asarray(a, dtype=np.float32)
    b = np.asarray(b, dtype=np.float32)
    a = (a - a.min()) / (a.max() - a.min() + 1e-6)
    b = (b - b.min()) / (b.max() - b.min() + 1e-6)
    hist, _, _ = np.histogram2d(a.ravel(), b.ravel(), bins=bins, range=[[0, 1], [0, 1]])
    pxy = hist / (np.sum(hist) + 1e-9)
    px = np.sum(pxy, axis=1, keepdims=True)
    py = np.sum(pxy, axis=0, keepdims=True)
    pxpy = px * py
    nz = pxy > 0
    return float(np.sum(pxy[nz] * np.log((pxy[nz] / (pxpy[nz] + 1e-9)) + 1e-9)))


def smoothness(flow):
    dy = np.abs(flow[:, 1:, :] - flow[:, :-1, :])
    dx = np.abs(flow[:, :, 1:] - flow[:, :, :-1])
    return float(np.mean(dx * dx) + np.mean(dy * dy))


def quiver_gain(flow, target_p95=2.0, max_gain=20.0):
    mag = np.sqrt(flow[0] ** 2 + flow[1] ** 2)
    p95 = float(np.percentile(mag, 95))
    if p95 <= 1e-8:
        return 1.0
    return float(np.clip(target_p95 / p95, 1.0, max_gain))


def plot_quiver(ax, flow, background, step=4, flip_y=True):
    h, w = flow.shape[1:]
    gain = quiver_gain(flow)
    y, x = np.mgrid[0:h:step, 0:w:step]
    fx = flow[0, ::step, ::step] * gain
    fy = flow[1, ::step, ::step] * gain
    if flip_y:
        fy = -fy
    ax.imshow(background, cmap='gray', origin='upper')
    ax.quiver(x, y, fx, fy, color='red', angles='xy', scale_units='xy', scale=1, headwidth=3, headlength=4, alpha=0.8)
    ax.set_xlim(0, w - 1)
    ax.set_ylim(h - 1, 0)
    ax.axis('off')
    return gain


def sensor_value(obj):
    if hasattr(obj, 'value'):
        value = obj.value
        value = value() if callable(value) else value
        return float(value)
    if hasattr(obj, 'sample'):
        value = obj.sample
        value = value() if callable(value) else value
        return float(value)
    if callable(obj):
        return float(obj())
    return float(obj)


def get_power_rails():
    try:
        from pynq.pmbus import get_rails
        return get_rails()
    except Exception:
        try:
            from pynq import get_rails
            return get_rails()
        except Exception:
            return None


def start_power_log(period_s=0.1, preferred=None):
    rails = get_power_rails()
    if not rails:
        return None, None
    keys = []
    if preferred and preferred in rails:
        keys.append(preferred)
    if not keys:
        for name in rails.keys():
            upper = name.upper()
            if 'FP' in upper or 'PL' in upper or 'INT' in upper or 'VCCINT' in upper:
                keys.append(name)
    if not keys:
        keys = list(rails.keys())
    keys = list(dict.fromkeys(keys))
    state = {'stop': False, 'rows': [], 'keys': keys, 't0': time.time()}
    def loop():
        while not state['stop']:
            row = {'time_sec': time.time() - state['t0']}
            total = 0.0
            for name in keys:
                try:
                    reading = sensor_value(getattr(rails[name], 'power', rails[name]))
                except Exception:
                    reading = float('nan')
                row[name] = reading
                if np.isfinite(reading):
                    total += reading
            row['total_w'] = total
            state['rows'].append(row)
            time.sleep(period_s)
    thread = threading.Thread(target=loop, daemon=True)
    thread.start()
    return state, thread


def stop_power_log(state, thread):
    if state is None or thread is None:
        return [], []
    state['stop'] = True
    thread.join(timeout=1.0)
    return state['rows'], state['keys']


def run_raw(inputs):
    input_bufs = []
    output_buf = None
    try:
        for arr in inputs:
            buf = allocate(shape=tuple(arr.shape), dtype=np.int8)
            np.copyto(buf, np.ascontiguousarray(arr, dtype=np.int8))
            if hasattr(buf, 'sync_to_device'):
                buf.sync_to_device()
            input_bufs.append(buf)
        output_buf = allocate(shape=output_shape, dtype=np.int8)
        job_id = dpu_runner.execute_async(input_bufs, [output_buf])
        dpu_runner.wait(job_id)
        if hasattr(output_buf, 'sync_from_device'):
            output_buf.sync_from_device()
        return np.array(output_buf, copy=True)
    finally:
        for buf in input_bufs:
            if hasattr(buf, 'freebuffer'):
                buf.freebuffer()
        if output_buf is not None and hasattr(output_buf, 'freebuffer'):
            output_buf.freebuffer()


def run_dpu_inference(moving_stack, fixed_stack):
    moving_stack = pad_stack_to_n(moving_stack, N_STACK)
    fixed_stack = pad_stack_to_n(fixed_stack, N_STACK)
    moving_input = moving_stack.transpose(1, 2, 0)[np.newaxis, ...].astype(np.float32)
    fixed_input = fixed_stack.transpose(1, 2, 0)[np.newaxis, ...].astype(np.float32)

    if len(input_tensors) == 2:
        moving_q = np.clip(np.round(moving_input * MOVING_SCALE), -128, 127).astype(np.int8)
        fixed_q = np.clip(np.round(fixed_input * FIXED_SCALE), -128, 127).astype(np.int8)
        name0 = str(input_tensors[0].name).lower()
        if 'moving' in name0:
            inputs = [moving_q, fixed_q]
        else:
            inputs = [fixed_q, moving_q]
    else:
        merged = np.concatenate([moving_stack, fixed_stack], axis=0).astype(np.float32)
        merged = merged.transpose(1, 2, 0)[np.newaxis, ...]
        scale = FIXED_SCALE or MOVING_SCALE
        inputs = [np.clip(np.round(merged * scale), -128, 127).astype(np.int8)]

    t0 = time.time()
    output = run_raw(inputs)
    dt = time.time() - t0

    flow = output.astype(np.float32) / OUTPUT_SCALE
    if flow.ndim == 4:
        flow = flow[0]
    if flow.shape[-1] == 2:
        flow = flow.transpose(2, 0, 1)
    return flow.astype(np.float32), dt


def infer_volume(moving_vol, fixed_vol, moving_seg=None, axis=0):
    depth = moving_vol.shape[axis]
    wr = WINDOW_RADIUS
    warped_vol = np.zeros_like(moving_vol)
    warped_seg_vol = np.zeros_like(moving_seg) if moving_seg is not None else None
    flow_list = []
    times = []

    for z in range(wr, depth - wr):
        moving_stack = extract_slice_stack(moving_vol, axis, z, wr, (INPUT_HEIGHT, INPUT_WIDTH))
        fixed_stack = extract_slice_stack(fixed_vol, axis, z, wr, (INPUT_HEIGHT, INPUT_WIDTH))
        flow, dt = run_dpu_inference(moving_stack, fixed_stack)
        flow_list.append(flow)
        times.append(dt)

        if axis == 0:
            moving_center = moving_vol[z]
            warped_vol[z] = warp_image(moving_center, flow)
            if moving_seg is not None:
                warped_seg_vol[z] = warp_image(moving_seg[z], flow, nearest=True)
        elif axis == 1:
            moving_center = moving_vol[:, z, :]
            warped_vol[:, z, :] = warp_image(moving_center, flow)
            if moving_seg is not None:
                warped_seg_vol[:, z, :] = warp_image(moving_seg[:, z, :], flow, nearest=True)
        else:
            moving_center = moving_vol[:, :, z]
            warped_vol[:, :, z] = warp_image(moving_center, flow)
            if moving_seg is not None:
                warped_seg_vol[:, :, z] = warp_image(moving_seg[:, :, z], flow, nearest=True)

        if (z - wr + 1) % 10 == 0:
            print(f'{z - wr + 1}/{depth - 2 * wr} slices, mean {np.mean(times) * 1000:.2f} ms')

    return warped_vol, np.array(flow_list, dtype=np.float32), np.array(times, dtype=np.float32), warped_seg_vol


In [ ]:
moving_vol = np.load(MR_VOLUME_PATH).astype(np.float32)
fixed_vol = np.load(CT_VOLUME_PATH).astype(np.float32)

has_seg = Path(MR_SEG_PATH).exists() and Path(CT_SEG_PATH).exists()
if has_seg:
    moving_seg = np.load(MR_SEG_PATH).astype(np.int16)
    fixed_seg = np.load(CT_SEG_PATH).astype(np.int16)
else:
    moving_seg = None
    fixed_seg = None

print('moving', moving_vol.shape, float(moving_vol.min()), float(moving_vol.max()))
print('fixed ', fixed_vol.shape, float(fixed_vol.min()), float(fixed_vol.max()))
print('segmentation:', has_seg)

z0 = moving_vol.shape[0] // 2
moving_stack0 = extract_slice_stack(moving_vol, 0, z0, WINDOW_RADIUS, (INPUT_HEIGHT, INPUT_WIDTH))
fixed_stack0 = extract_slice_stack(fixed_vol, 0, z0, WINDOW_RADIUS, (INPUT_HEIGHT, INPUT_WIDTH))
for _ in range(WARMUP_RUNS):
    _flow, _ = run_dpu_inference(moving_stack0, fixed_stack0)
flow0, t0 = run_dpu_inference(moving_stack0, fixed_stack0)
print('center slice time ms:', round(t0 * 1000, 3))
print('flow range:', float(flow0.min()), float(flow0.max()))


In [ ]:
power_state, power_thread = start_power_log(POWER_PERIOD_S, PREFERRED_POWER_RAIL)
wall_t0 = time.time()
warped_vol, flow_vol, inf_times, warped_seg_vol = infer_volume(moving_vol, fixed_vol, moving_seg, axis=0)
wall_time = time.time() - wall_t0
power_rows, power_keys = stop_power_log(power_state, power_thread)

print('slices:', len(inf_times))
print('mean ms:', round(float(np.mean(inf_times) * 1000), 3))
print('wall time s:', round(float(wall_time), 3))
if power_rows:
    print('power rails:', power_keys)
    print('power samples:', len(power_rows))
else:
    print('power rails not available')


In [ ]:
valid = slice(WINDOW_RADIUS, moving_vol.shape[0] - WINDOW_RADIUS)
mi_before = mutual_info(moving_vol[valid], fixed_vol[valid])
mi_after = mutual_info(warped_vol[valid], fixed_vol[valid])
smooth_loss = float(np.mean([smoothness(flow) for flow in flow_vol])) if len(flow_vol) else 0.0

results = {
    'slices': int(len(inf_times)),
    'warmup_runs': int(WARMUP_RUNS),
    'mean_time_ms': float(np.mean(inf_times) * 1000),
    'std_time_ms': float(np.std(inf_times) * 1000),
    'min_time_ms': float(np.min(inf_times) * 1000),
    'max_time_ms': float(np.max(inf_times) * 1000),
    'dpu_time_total_s': float(np.sum(inf_times)),
    'wall_time_s': float(wall_time),
    'throughput_slices_per_s': float(len(inf_times) / wall_time) if wall_time > 0 else None,
    'mi_before': float(mi_before),
    'mi_after': float(mi_after),
    'mi_loss_after': float(-mi_after),
    'smooth_loss': float(smooth_loss),
    'output_scale': float(OUTPUT_SCALE),
}

if has_seg:
    dice_before = dice_score(moving_seg[valid], fixed_seg[valid], SEG_LABELS)
    dice_after = dice_score(warped_seg_vol[valid].astype(np.int16), fixed_seg[valid], SEG_LABELS)
    dice_loss_after = None if dice_after is None else float(1.0 - dice_after)
    total_loss = (dice_loss_after if dice_loss_after is not None else 0.0) + 0.5 * (-mi_after) + 0.1 * smooth_loss
    results['dice_before'] = None if dice_before is None else float(dice_before)
    results['dice_after'] = None if dice_after is None else float(dice_after)
    results['dice_loss_after'] = dice_loss_after
    results['total_loss'] = float(total_loss)
else:
    results['total_loss'] = float(0.5 * (-mi_after) + 0.1 * smooth_loss)

if power_rows:
    total_power = np.array([row['total_w'] for row in power_rows], dtype=np.float32)
    results['power_mean_w'] = float(np.nanmean(total_power))
    results['power_peak_w'] = float(np.nanmax(total_power))
    results['power_samples'] = int(len(power_rows))
    rail_means = {}
    for key in power_keys:
        rail_vals = np.array([row.get(key, np.nan) for row in power_rows], dtype=np.float32)
        rail_means[key] = float(np.nanmean(rail_vals))
    results['power_rail_mean_w'] = rail_means
else:
    results['power_mean_w'] = None
    results['power_peak_w'] = None
    results['power_samples'] = 0
    results['power_rail_mean_w'] = {}

Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)
Path(OUTPUT_DIR, 'metrics.json').write_text(json.dumps(results, indent=2))

with open(Path(OUTPUT_DIR) / 'slice_times_ms.csv', 'w', encoding='utf-8') as f:
    f.write('slice,time_ms\n')
    for i, t in enumerate(inf_times, start=WINDOW_RADIUS):
        f.write(f'{i},{float(t * 1000):.6f}\n')

if power_rows:
    columns = ['time_sec', 'total_w'] + list(power_keys)
    with open(Path(OUTPUT_DIR) / 'power.csv', 'w', encoding='utf-8') as f:
        f.write(','.join(columns) + '\n')
        for row in power_rows:
            vals = []
            for key in columns:
                val = row.get(key, '')
                if isinstance(val, float):
                    vals.append(f'{val:.6f}')
                else:
                    vals.append(str(val))
            f.write(','.join(vals) + '\n')

z_show = moving_vol.shape[0] // 2
z_show = min(max(z_show, WINDOW_RADIUS), moving_vol.shape[0] - WINDOW_RADIUS - 1)
flow_show = flow_vol[z_show - WINDOW_RADIUS]
moving_show = moving_vol[z_show]
fixed_show = fixed_vol[z_show]
warped_show = warped_vol[z_show]
flow_bg = cv2.resize(moving_show, (flow_show.shape[2], flow_show.shape[1]), interpolation=cv2.INTER_LINEAR)

fig, axes = plt.subplots(2, 2, figsize=(11, 9))
axes[0, 0].imshow(moving_show, cmap='gray')
axes[0, 0].set_title('Moving')
axes[0, 0].axis('off')
axes[0, 1].imshow(fixed_show, cmap='gray')
axes[0, 1].set_title('Fixed')
axes[0, 1].axis('off')
axes[1, 0].imshow(warped_show, cmap='gray')
axes[1, 0].set_title('Warped')
axes[1, 0].axis('off')
q_gain = plot_quiver(axes[1, 1], flow_show, flow_bg, step=4, flip_y=QUIVER_FLIP_Y)
axes[1, 1].set_title(f'Flow x{q_gain:.1f}')
plt.tight_layout()
plt.savefig(Path(OUTPUT_DIR) / 'preview.png', dpi=150)
plt.show()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].plot(np.arange(WINDOW_RADIUS, WINDOW_RADIUS + len(inf_times)), inf_times * 1000, color='tab:blue')
axes[0].set_title('Slice Time')
axes[0].set_xlabel('Slice')
axes[0].set_ylabel('ms')
axes[0].grid(alpha=0.3)
if power_rows:
    power_t = np.array([row['time_sec'] for row in power_rows], dtype=np.float32)
    power_v = np.array([row['total_w'] for row in power_rows], dtype=np.float32)
    axes[1].plot(power_t, power_v, color='tab:red')
    axes[1].set_title('Power')
    axes[1].set_xlabel('s')
    axes[1].set_ylabel('W')
    axes[1].grid(alpha=0.3)
else:
    axes[1].text(0.5, 0.5, 'Power rails not available', ha='center', va='center', fontsize=12)
    axes[1].set_axis_off()
plt.tight_layout()
plt.savefig(Path(OUTPUT_DIR) / 'timing_power.png', dpi=150)
plt.show()

print(json.dumps(results, indent=2))


In [ ]:
del dpu_runner
del graph
print('done')
